# Metaflora Incubus v1 — CPU recovery
One cell restores the trained adapter, exports Q5_K_M, runs the same committed 48-case benchmark on CPU and syncs the private artifact. Training is not repeated.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

trusted_code_revision = "15cd6c967239769928c709031667adb2ca95ab35"
repository = "/content/metaflora-incubus"
os.chdir("/content")
shutil.rmtree(repository, ignore_errors=True)
subprocess.run(["git", "init", repository], check=True)
subprocess.run(["git", "-C", repository, "remote", "add", "origin", "https://github.com/metaflora-app/metaflora-incubus.git"], check=True)
subprocess.run(["git", "-C", repository, "fetch", "--depth", "1", "origin", trusted_code_revision], check=True)
subprocess.run(["git", "-C", repository, "checkout", "--detach", "FETCH_HEAD"], check=True)
checked_out = subprocess.run(["git", "-C", repository, "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
if checked_out != trusted_code_revision:
    raise RuntimeError("trusted code revision verification failed")
subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", f"{repository}/requirements/recovery-linux.lock"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", repository], check=True)
source_root = f"{repository}/src"
if source_root not in sys.path:
    sys.path.insert(0, source_root)
os.chdir(repository)
from google.colab import userdata
from metaflora_incubus.cloud_bootstrap import decrypt_cloud_bootstrap, install_cloud_bootstrap
encoded_bootstrap = userdata.get("INCUBUS_BOOTSTRAP")
if not isinstance(encoded_bootstrap, str) or not encoded_bootstrap.strip():
    raise RuntimeError("INCUBUS_BOOTSTRAP is empty")
bootstrap_payload = decrypt_cloud_bootstrap(Path("configs/cloud/bootstrap-v1.enc").read_bytes(), encoded_bootstrap)
install_cloud_bootstrap(bootstrap_payload, home=Path.home(), environment=os.environ)
runtime_environment = dict(os.environ)
subprocess.run([sys.executable, "scripts/recover_free_gpu.py", "--cpu-fallback", "--run-id", "incubus-v1-run", "--parameter-count", os.environ["INCUBUS_PARAMETER_COUNT"], "--checkpoint-location", "metaflora/incubus-checkpoints", "--checkpoint-branch", "incubus-training-v1"], check=True, env=runtime_environment)
